# Setup

In [1]:
# connect to drive
from google.colab import drive
drive.mount('/content/drive')
import os
from pathlib import Path
import shutil
import json
# Google Colab: instalar Playwright, Chromium y dependencias del sistema.
!pip -q install playwright
!playwright install --with-deps chromium
from urllib.parse import urlparse, unquote
import html
import requests
import pandas as pd
from playwright.async_api import async_playwright


Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 17.0 MB/s eta 0:00:00
Installing dependencies...
Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:5 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [105 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,616 kB]
Hit:12 https://ppa.launchpadcontent.net/u

In [2]:
def load_json(path):
    with open(path, 'r') as f:
        return json.load(f)

def save_json(path, data):
    with open(path, 'w') as f:
        json.dump(data, f, indent=4)


# Scrapping from Wildlife


Copia aquí la cookie de sesión `connect.sid` de `app.wildlifeinsights.org`.

En Chrome puedes verla en:

**DevTools → Application → Cookies → https://app.wildlifeinsights.org**

La sesión puede expirar. Si deja de funcionar, vuelve a copiar el valor actualizado.

In [3]:
# Scrapping from Wildlife# Pega solamente el VALOR de connect.sid, no "connect.sid=" completo.
CONNECT_SID = "s%3AcD7M65eq7_gL9o-Ap2crT7sBJFhOBPcp.toZ28eakOdh%2BPzzL%2FZUqZFXbXI8%2BZZRYlQpdRCbrYCY"

OUTPUT_FOLDER = Path("/content/data/15")
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

print("Carpeta de salida:", OUTPUT_FOLDER)

Carpeta de salida: /content/data/15


In [5]:
def filename_from_signed_url(image_url: str, fallback: str = "image.jpg") -> str:
    path = unquote(urlparse(image_url).path)
    filename = Path(path).name
    return filename if filename else fallback


def download_signed_url(image_url: str, save_path: Path) -> None:
    image_url = html.unescape(image_url)

    response = requests.get(
        image_url,
        timeout=60,
        stream=True,
    )
    response.raise_for_status()

    content_type = response.headers.get("Content-Type", "").lower()

    if content_type and not content_type.startswith("image/"):
        raise ValueError(
            f"La URL no devolvió una imagen. Content-Type: {content_type}"
        )

    with save_path.open("wb") as file:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                file.write(chunk)

async def open_wildlife_browser(playwright):
    browser = await playwright.chromium.launch(
        headless=True,
        args=[
            "--no-sandbox",
            "--disable-dev-shm-usage",
        ],
    )

    context = await browser.new_context(
        user_agent=(
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 Chrome/120.0 Safari/537.36"
        )
    )

    if not CONNECT_SID or CONNECT_SID == "PEGA_AQUI_EL_VALOR_DE_CONNECT.SID":
        raise ValueError(
            "Debes pegar el valor de la cookie connect.sid en CONNECT_SID."
        )

    await context.add_cookies([
        {
            "name": "connect.sid",
            "value": CONNECT_SID,
            "domain": "app.wildlifeinsights.org",
            "path": "/",
            "secure": True,
            "httpOnly": True,
            "sameSite": "Lax",
        }
    ])

    page = await context.new_page()

    return browser, context, page

async def extract_image_url(page, page_url: str) -> str:
    page_url = str(page_url).strip().rstrip("+")

    response = await page.goto(
        page_url,
        wait_until="domcontentloaded",
        timeout=60_000,
    )

    if response is not None and response.status >= 400:
        raise RuntimeError(f"Wildlife Insights respondió HTTP {response.status}")

    image = page.locator("div.c-public-image img").first

    await image.wait_for(
        state="attached",
        timeout=30_000,
    )

    image_url = await image.get_attribute("src")

    if not image_url:
        raise RuntimeError("Se encontró el <img>, pero no tiene atributo src.")

    return html.unescape(image_url)


Prueba una sola imagen para ver si funciona correctamente y las cookies estan correctamente configuradas.

In [7]:
# Copia cualquier url de la columna location de tu archivo image.csv
PAGE_URL = "https://app.wildlifeinsights.org/download/2025748/project/2003870/data-files/9c2b7afc-1cc1-4bda-bfea-a19d2b61b15c"

async with async_playwright() as playwright:
    browser, context, page = await open_wildlife_browser(playwright)

    try:
        image_url = await extract_image_url(page, PAGE_URL)

        print("URL firmada encontrada:")
        print(image_url)

        filename = filename_from_signed_url(image_url)
        save_path = OUTPUT_FOLDER / filename

        download_signed_url(image_url, save_path)

        print("\nImagen guardada en:")
        print(save_path)

    finally:
        await browser.close()


URL firmada encontrada:
https://storage.googleapis.com/437283855702_2003870_425_p276__main/deployment%2F2102497%2F9c2b7afc-1cc1-4bda-bfea-a19d2b61b15c_500.jpg?GoogleAccessId=wi-api%40wildlifeinsights-external.iam.gserviceaccount.com&Expires=1786201698&Signature=cJQ%2FtLqEsUqlp6xd0VWQ9nn5uBjTv%2FbEmg4Mr6a9iZAHfZGO0BD85Zl8eyKpjSV1cjLEqq9aEgGpMnBQHqbpXQzaBB5A9ry4WSZx2bh6ZpNbXYtpb%2FUxUB%2FUJztAIAazqVjnA6X9JezylmHTuflubpTE%2FqCDp46m3BAKhZxX1akVSlgNHOF6zjQLsAFY03IipEuG38ZNzfGpARRY6oNqVBej%2BsCGNSqoWaIBrwdY2DY1qVOH8oYR30v5LpmJmmDxmG4eSPIb4%2B2u17kOiVQ4cojwOxhbrVxEq7RTlV%2FeBPqC4vjsqLSr5IVnSVW8IwVfUUFoxw5SmbH6hcBYp4YNVA%3D%3D


HTTPError: 404 Client Error: Not Found for url: https://storage.googleapis.com/437283855702_2003870_425_p276__main/deployment%2F2102497%2F9c2b7afc-1cc1-4bda-bfea-a19d2b61b15c_500.jpg?GoogleAccessId=wi-api%40wildlifeinsights-external.iam.gserviceaccount.com&Expires=1786201698&Signature=cJQ%2FtLqEsUqlp6xd0VWQ9nn5uBjTv%2FbEmg4Mr6a9iZAHfZGO0BD85Zl8eyKpjSV1cjLEqq9aEgGpMnBQHqbpXQzaBB5A9ry4WSZx2bh6ZpNbXYtpb%2FUxUB%2FUJztAIAazqVjnA6X9JezylmHTuflubpTE%2FqCDp46m3BAKhZxX1akVSlgNHOF6zjQLsAFY03IipEuG38ZNzfGpARRY6oNqVBej%2BsCGNSqoWaIBrwdY2DY1qVOH8oYR30v5LpmJmmDxmG4eSPIb4%2B2u17kOiVQ4cojwOxhbrVxEq7RTlV%2FeBPqC4vjsqLSr5IVnSVW8IwVfUUFoxw5SmbH6hcBYp4YNVA%3D%3D

Descargamos todas las imagenes

In [9]:
if not os.path.exists("/content/data"):
    os.mkdir("/content/data")

# Importante: rellenar esta lista segun la clase a rescatar
classes = ["zorro_grilla","zorro_culpeo"]

for cls in classes:
    if not os.path.exists(f"/content/data/{cls}"):
        os.mkdir(f"/content/data/{cls}")

csv_paths = [Path(f"/content/data/{cls}/images.csv") for cls in classes]
print(csv_paths)

[PosixPath('/content/data/zorro_grilla/images.csv'), PosixPath('/content/data/zorro_culpeo/images.csv')]


In [10]:
async def download_from_dataframe(
    df: pd.DataFrame,
    output_folder: Path,
    location_column: str = "location",
):
    if location_column not in df.columns:
        raise ValueError(
            f"No existe la columna '{location_column}' en el DataFrame."
        )

    urls = df[location_column].dropna().astype(str)

    downloaded = 0
    skipped = 0
    failed = []

    async with async_playwright() as playwright:
        browser, context, page = await open_wildlife_browser(playwright)

        try:
            for index, page_url in urls.items():
                try:
                    image_url = await extract_image_url(page, page_url)

                    filename = filename_from_signed_url(
                        image_url,
                        fallback=f"image_{index}.jpg",
                    )

                    save_path = output_folder / filename

                    if save_path.exists() and save_path.stat().st_size > 0:
                        skipped += 1
                        continue

                    download_signed_url(image_url, save_path)

                    downloaded += 1

                    if downloaded % 25 == 0:
                        print(f"Descargadas: {downloaded}")

                except Exception as error:
                    failed.append({
                        "index": index,
                        "location": page_url,
                        "error": str(error),
                    })

        finally:
            await browser.close()

    print("\nProceso terminado")
    print("Descargadas:", downloaded)
    print("Ya existentes:", skipped)
    print("Fallidas:", len(failed))

    failures_df = pd.DataFrame(failed)

    if not failures_df.empty:
        failures_path = output_folder / "failed_downloads.csv"
        failures_df.to_csv(failures_path, index=False)

        print("Errores guardados en:")
        print(failures_path)

    return failures_df

In [11]:
# Comenzamos la descarga, todo se guardara en /data.
for csv_path in csv_paths:
    output_folder = Path(csv_path).parent
    output_folder = output_folder.parent / output_folder.name
    csv_files = [f for f in output_folder.iterdir() if f.suffix == ".csv"]
    for csv_file in csv_files:
        df = pd.read_csv(csv_file)
        failures_df = await download_from_dataframe(
            df=df,
            output_folder=output_folder,
            location_column="location",
        )


Proceso terminado
Descargadas: 17
Ya existentes: 0
Fallidas: 582
Errores guardados en:
/content/data/zorro_grilla/failed_downloads.csv

Proceso terminado
Descargadas: 3
Ya existentes: 0
Fallidas: 740
Errores guardados en:
/content/data/zorro_culpeo/failed_downloads.csv


# Upload Wildlife files to drive

In [12]:
DEST_PATH = Path('drive/MyDrive/ECHO/Data/Training/')
if not os.path.exists(DEST_PATH):
    raise Exception('Destination path does not exist')

In [14]:
id_to_specie = load_json(DEST_PATH / 'id_to_specie.json')
specie_to_id = {v: k for k, v in id_to_specie.items()}
#print(list(id_to_specie.items()))
# create the folder data if doesn't exist
SOURCE_PATH = Path("/content/data")

for dir in SOURCE_PATH.iterdir():
    if dir.is_file():
        continue
    if dir.name.startswith('.') or dir.name in ['drive', 'sample_data']:
        continue
    files_count = len([f for f in dir.iterdir() if not f.name.endswith('.csv')])
    print(dir.name, 'files count: ', files_count)
    exists_in_json = specie_to_id.get(dir.name)
    if not exists_in_json:
        print("",dir.name, "doesn't exist in /Training")
        print(" Creating class directory")
        max_cls = int(max([int(x) for x in id_to_specie.keys()]))
        new_cls = str(max_cls + 1)
        id_to_specie[new_cls] = dir.name
        save_json(DEST_PATH / 'id_to_specie.json', id_to_specie)
    id_to_specie = load_json(DEST_PATH / 'id_to_specie.json')
    specie_to_id = {v: k for k, v in id_to_specie.items()}
    specie_cls = specie_to_id[dir.name]
    print(' Specie class: ', specie_cls)
    unprocessed_dir = DEST_PATH / specie_cls / 'unprocessed'
    if not unprocessed_dir.exists():
        print(' Creating unprocessed directory')
        unprocessed_dir.mkdir(parents=True)
    files = [
        f for f in dir.iterdir()
        if f.name.endswith('.JPG') or
            f.name.endswith('.jpg') or
            f.name.endswith('.png') or
            f.name.endswith('.PNG')
    ]
    print(' Moving files to unprocessed directory')
    for f in files:
        shutil.move(f, unprocessed_dir / f.name)
    print(' Done')


zorro_chilla files count:  17
 Specie class:  18
 Creating unprocessed directory
 Moving files to unprocessed directory
 Done
zorro_culpeo files count:  3
 Specie class:  14
 Creating unprocessed directory
 Moving files to unprocessed directory
 Done


In [16]:
# Check total of files for each class
id_to_specie = load_json(DEST_PATH / 'id_to_specie.json')
for dir in DEST_PATH.iterdir():
    if dir.is_file():
        continue
    specie = id_to_specie[dir.name]
    print("Class ID: ", dir.name, " / Specie: ", specie)
    processed_files = len([f for f in dir.iterdir() if f.is_file()])
    print("- Processed files: ", processed_files)
    unprocessed_dir = dir / 'unprocessed'
    if not unprocessed_dir.exists():
        continue
    unprocessed_files = len([f for f in unprocessed_dir.iterdir() if f.is_file()])
    print("- Unprocessed files: ", unprocessed_files)
    print("- Total files: ", processed_files + unprocessed_files)



Class ID:  8  / Specie:  guanaco
- Processed files:  251
Class ID:  9  / Specie:  huemul
- Processed files:  243
Class ID:  3  / Specie:  guina
- Processed files:  328
- Unprocessed files:  155
- Total files:  483
Class ID:  1  / Specie:  pudu
- Processed files:  463
Class ID:  6  / Specie:  caballo
- Processed files:  178
Class ID:  2  / Specie:  puma
- Processed files:  471
Class ID:  4  / Specie:  chucao
- Processed files:  457
- Unprocessed files:  389
- Total files:  846
Class ID:  0  / Specie:  hued_hued
- Processed files:  474
Class ID:  7  / Specie:  chingue
- Processed files:  197
Class ID:  5  / Specie:  zorzal
- Processed files:  448
Class ID:  13  / Specie:  vaca
- Processed files:  191
Class ID:  11  / Specie:  oveja
- Processed files:  196
Class ID:  14  / Specie:  zorro_culpeo
- Processed files:  201
- Unprocessed files:  3
- Total files:  204
Class ID:  10  / Specie:  liebre_europea
- Processed files:  202
Class ID:  15  / Specie:  vison_americano
- Processed files:  19